In [2]:
import pandas as pd
import os

caminho_excel = '../data/raw/mapbiomas_desmatamento.xlsx'

print("⏳ Lendo o arquivo gigante do MapBiomas (isso pode levar uns 2 minutos)...")
df_mapbiomas = pd.read_excel(caminho_excel, sheet_name='DEFORESTATION')

# 1. A Correção: Forçar todos os nomes de colunas a serem texto (String)
df_mapbiomas.columns = df_mapbiomas.columns.astype(str)

print("✅ Arquivo carregado! Filtrando apenas Vitória - ES...")

# 2. O Filtro de Vitória
df_vitoria = df_mapbiomas[(df_mapbiomas['state'] == 'Espírito Santo') & 
                          (df_mapbiomas['municipality'] == 'Vitória')].copy()

# 3. Os Anos agora como TEXTO (com aspas)
anos_interesse = ['2021', '2022', '2023', '2024']
colunas_identificacao = ['state', 'municipality'] # Simplificamos as colunas de ID para não dar erro

# Isolar as colunas
colunas_finais = colunas_identificacao + anos_interesse

# Se por acaso os anos não existirem, o script avisa em vez de quebrar
colunas_disponiveis = [col for col in colunas_finais if col in df_vitoria.columns]
df_vitoria_filtrado = df_vitoria[colunas_disponiveis]

# 4. Derreter (Melt) de Larga para Longa
df_derretido = pd.melt(df_vitoria_filtrado, 
                       id_vars=colunas_identificacao, 
                       var_name='ANO', 
                       value_name='AREA_DESMATADA_HA')

# 5. Agregação Final (Soma total de hectares desmatados por ano em Vitória)
df_anual = df_derretido.groupby('ANO')['AREA_DESMATADA_HA'].sum().reset_index()

# 6. Salvar
caminho_salvamento = '../data/raw/mapbiomas_vitoria_agregado_2021_2024.csv'
df_anual.to_csv(caminho_salvamento, index=False)

print(f"\n🎉 Sucesso! Tabela de desmatamento salva em: {caminho_salvamento}")
print("\nEvolução da supressão vegetal (em hectares) na nossa ilha:")
display(df_anual)

⏳ Lendo o arquivo gigante do MapBiomas (isso pode levar uns 2 minutos)...
✅ Arquivo carregado! Filtrando apenas Vitória - ES...

🎉 Sucesso! Tabela de desmatamento salva em: ../data/raw/mapbiomas_vitoria_agregado_2021_2024.csv

Evolução da supressão vegetal (em hectares) na nossa ilha:


,ANO,AREA_DESMATADA_HA
0,2021,3.779856
1,2022,16.546257
2,2023,7.726314
3,2024,13.607769
